# 03 - Addition of Basic Autoencoder


## 🌐 Connect Colab to Google Drive

In [ ]:
from google.colab import drive

drive.mount("/gdrive")
%cd /gdrive/My Drive/[2024-2025] AN2DL

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive/My Drive/[2024-2025] AN2DL


## ⚙️ Import Libraries

In [ ]:
import random
import seaborn as sns
import DeepLearningLib as dll

In [ ]:
import os
from datetime import datetime

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras as tfk
from tensorflow.keras import layers as tfkl

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tfk.__version__}")
print(f"GPU devices: {len(tf.config.list_physical_devices('GPU'))}")

TensorFlow version: 2.17.1
Keras version: 3.5.0
GPU devices: 1


## ⏳ Load the Data

In [ ]:
data = np.load("mars_for_students.npz")

training_set = data["training_set"]
X_train = training_set[:, 0]
y_train = training_set[:, 1]

X_test = data["test_set"]

print(f"Training X shape: {X_train.shape}")
print(f"Training y shape: {y_train.shape}")
print(f"Test X shape: {X_test.shape}")

Training X shape: (2615, 64, 128)
Training y shape: (2615, 64, 128)
Test X shape: (10022, 64, 128)


## 🛠️ Train and Save the Model

In [ ]:
# Add color channel and rescale pixels between 0 and 1
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

input_shape = X_train.shape[1:]
num_classes = len(np.unique(y_train))

print(f"Input shape: {input_shape}")
print(f"Number of classes: {num_classes}")

Input shape: (64, 128, 1)
Number of classes: 5


In [ ]:
y_train = y_train[..., np.newaxis]
y_train.shape

(2615, 64, 128, 1)

In [ ]:
# Set batch size for training
BATCH_SIZE = 64

# Set learning rate for the optimiser
LEARNING_RATE = 1e-3

# Set early stopping patience threshold
PATIENCE = 30

# Set maximum number of training epochs
EPOCHS = 1000

# Set data split size for training and validation
SPLITS_SIZE = 300

In [ ]:
# Devide the train data reservig 300 samples for validation

train_img, val_img, train_lbl, val_lbl = train_test_split(
    X_train, y_train, test_size=300, random_state=42
)
print("Data splitted!")

print(f"\nNumber of images:")
print(f"Train: {len(train_img)}")
print(f"Validation: {len(val_img)}")

Data splitted!

Number of images:
Train: 2315
Validation: 300


In [ ]:
# Functions used for augmentation

@tf.function
def random_horizontal_flip(image, label, seed=None):
    """Consistent random horizontal flip."""
    if seed is None:
        seed = np.random.randint(0, 1000000)
    flip_prob = tf.random.uniform([], seed=seed)
    image = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.flip_left_right(image),
        lambda: image
    )
    label = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.flip_left_right(label),
        lambda: label
    )
    return image, label

@tf.function
def random_vertical_flip(image, label, seed=None):
    """Consistent random vertical flip."""
    if seed is None:
        seed = np.random.randint(0, 1000000)
    flip_prob = tf.random.uniform([], seed=seed)
    image = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.flip_up_down(image),
        lambda: image
    )
    label = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.flip_up_down(label),
        lambda: label
    )
    return image, label

@tf.function
def random_brightness(image, label, seed=None):
    """Consistent random brightness flip."""
    if seed is None:
        seed = np.random.randint(0, 1000000)
    flip_prob = tf.random.uniform([], seed=seed)
    image = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.stateless_random_brightness(image, max_delta=.2, seed=seed),
        lambda: image
    )
    # label = tf.cond(
    #     flip_prob > 0.5,
    #     lambda: tf.image.stateless_random_brightness(image, max_delta=.2, seed=seed),
    #     lambda: label
    # )
    return image, label

@tf.function
def random_contrast(image, label, seed=None):
    """Consistent random contrast flip."""
    if seed is None:
        seed = np.random.randint(0, 1000000)
    flip_prob = tf.random.uniform([], seed=seed)
    image = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.stateless_random_contrast(image, lower=0.8, upper=1.2, seed=seed),
        lambda: image
    )
    # label = tf.cond(
    #     flip_prob > 0.5,
    #     lambda: tf.image.stateless_random_contrast(image, lower=0.8, upper=1.2, seed=seed),
    #     lambda: label
    # )
    return image, label

@tf.function
def random_crop(image, label, seed=None):
    """Consistent random crop flip."""
    if seed is None:
        seed = np.random.randint(0, 1000000)
    flip_prob = tf.random.uniform([], seed=seed)
    image = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.stateless_random_crop(image, size=(32, 64, 1), seed=seed),
        lambda: image
    )
    label = tf.cond(
        flip_prob > 0.5,
        lambda: tf.image.stateless_random_crop(image, size=(32, 64, 1), seed=seed),
        lambda: label
    )
    return image, label

In [ ]:
def unet_block(input_tensor, filters, kernel_size=3, activation='relu', stack=2, name=''):
    # Initialise the input tensor
    x = input_tensor

    # Apply a sequence of Conv2D, Batch Normalisation, and Activation layers for the specified number of stacks
    for i in range(stack):
        x = tfkl.Conv2D(filters, kernel_size=kernel_size, padding='same', name=name + 'conv' + str(i + 1))(x)
        x = tfkl.BatchNormalization(name=name + 'bn' + str(i + 1))(x)
        x = tfkl.Activation(activation, name=name + 'activation' + str(i + 1))(x)

    # Return the transformed tensor
    return x

### No Augmentation

In [ ]:
def make_dataset(X, y, batch_size=64, seed=None):
    """
    Create a memory-efficient TensorFlow dataset that merely take data (do not apply augmentation)
    """
    # Create dataset from file paths
    dataset = tf.data.Dataset.from_tensor_slices((X, y))

    # Batch the data
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
# Create the datasets
print("Creating datasets...")
train_dataset = make_dataset(
    train_img, train_lbl,
    batch_size=BATCH_SIZE,
    seed=42
)

val_dataset = make_dataset(
    val_img, val_lbl,
    batch_size=BATCH_SIZE,
)

print("Datasets created!")

# Check the shape of the data
for images, labels in train_dataset.take(1):
    input_shape = images.shape[1:]
    print(f"\nInput shape: {input_shape}")
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Labels dtype:", labels.dtype)
    break

Creating datasets...
Datasets created!

Input shape: (64, 128, 1)
Images shape: (64, 64, 128, 1)
Labels shape: (64, 64, 128)
Labels dtype: <dtype: 'float64'>


In [ ]:
# Model Creation
seed = 42

tf.random.set_seed(seed)
input_layer = tfkl.Input(shape=input_shape, name='input_layer')

# Downsampling path
down_block_1 = unet_block(input_layer, 32, name='down_block1_')
d1 = tfkl.MaxPooling2D()(down_block_1)

down_block_2 = unet_block(d1, 64, name='down_block2_')
d2 = tfkl.MaxPooling2D()(down_block_2)

# Bottleneck
bottleneck = unet_block(d2, 128, name='bottleneck')

# Upsampling path
u1 = tfkl.UpSampling2D()(bottleneck)
u1 = tfkl.Concatenate()([u1, down_block_2])
u1 = unet_block(u1, 64, name='up_block1_')

u2 = tfkl.UpSampling2D()(u1)
u2 = tfkl.Concatenate()([u2, down_block_1])
u2 = unet_block(u2, 32, name='up_block2_')

# Output Layer
output_layer = tfkl.Conv2D(num_classes, kernel_size=1, padding='same', activation="softmax", name='output_layer')(u2)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer, name='UNet')

# Define the MeanIoU ignoring the background class
mean_iou = tfk.metrics.MeanIoU(num_classes=num_classes, ignore_class=0, sparse_y_pred=False)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=[mean_iou, "accuracy"])

model.summary()

Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 64, 128, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv1         │ (None, 64, 128, 32)    │            320 │ input_layer[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn1           │ (None, 64, 128, 32)    │            128 │ down_block1_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation1   │ (None, 64, 128, 32)    │              0 │ down_block1_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv2         │ (None, 64, 128, 32)    │          9,248 │ down_block1_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn2           │ (None, 64, 128, 32)    │            128 │ down_block1_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation2   │ (None, 64, 128, 32)    │              0 │ down_block1_bn2[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d             │ (None, 32, 64, 32)     │              0 │ down_block1_activatio… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv1         │ (None, 32, 64, 64)     │         18,496 │ max_pooling2d[0][0]    │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn1           │ (None, 32, 64, 64)     │            256 │ down_block2_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activation1   │ (None, 32, 64, 64)     │              0 │ down_block2_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv2         │ (None, 32, 64, 64)     │         36,928 │ down_block2_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn2           │ (None, 32, 64, 64)     │            256 │ down_block2_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activatio

 Total params: 473,669 (1.81 MB)

 Trainable params: 472,389 (1.80 MB)

 Non-trainable params: 1,280 (5.00 KB)

In [ ]:
# Setup callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=PATIENCE,
    restore_best_weights=True
)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
).history

Epoch 1/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 47s 618ms/step - accuracy: 0.3642 - loss: 1.4889 - mean_io_u: 0.1561 - val_accuracy: 0.1884 - val_loss: 2.2801 - val_mean_io_u: 0.0513
Epoch 2/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 45s 113ms/step - accuracy: 0.5134 - loss: 1.1778 - mean_io_u: 0.2594 - val_accuracy: 0.1884 - val_loss: 3.6365 - val_mean_io_u: 0.0513
Epoch 3/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.5451 - loss: 1.1207 - mean_io_u: 0.2798 - val_accuracy: 0.1884 - val_loss: 4.4068 - val_mean_io_u: 0.0513
Epoch 4/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 113ms/step - accuracy: 0.5945 - loss: 1.0223 - mean_io_u: 0.3174 - val_accuracy: 0.1884 - val_loss: 3.8912 - val_mean_io_u: 0.0513
Epoch 5/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 117ms/step - accuracy: 0.6209 - loss: 0.9635 - mean_io_u: 0.3386 - val_accuracy: 0.1885 - val_loss: 3.5555 - val_mean_io_u: 0.0513
Epoch 6/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 120ms/step - accuracy: 0.6402 - loss: 0.9102 - mean_io_u: 0.3567 - val_accuracy: 0.18

In [ ]:
timestep_str = datetime.now().strftime("%y%m%d_%H%M%S")
model_filename = f"model_{timestep_str}.keras"
model.save(model_filename)
del model

print(f"Model saved to {model_filename}")

Model saved to model_241202_111634.keras


### Train with Augmentation One (Only vertical and horizontal flips)

In [ ]:
def make_dataset_augOne(image_paths, label_paths, batch_size, shuffle=False, augment=False, seed=None):
    """
    Create a memory-efficient TensorFlow dataset that beyond taking the data apply augmentation.
    Augmentation applied in this case: horizontal and vertical flips.
    """
    # Create dataset from file paths
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, label_paths))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=batch_size * 2, seed=seed)

    if augment:
        dataset = dataset.map(
            lambda x, y: random_horizontal_flip(x, y, seed=seed),
            num_parallel_calls=tf.data.AUTOTUNE
        )

        dataset = dataset.map(
            lambda x, y: random_vertical_flip(x, y, seed=seed),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # Batch the data
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
# Create the datasets
print("Creating datasets...")
train_dataset = make_dataset_augOne(
    train_img, train_lbl,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    seed=42
)

val_dataset = make_dataset_augOne(
    val_img, val_lbl,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Datasets created!")

# Check the shape of the data
for images, labels in train_dataset.take(1):
    input_shape = images.shape[1:]
    print(f"\nInput shape: {input_shape}")
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Labels dtype:", labels.dtype)
    break

Creating datasets...
Datasets created!

Input shape: (64, 128, 1)
Images shape: (64, 64, 128, 1)
Labels shape: (64, 64, 128, 1)
Labels dtype: <dtype: 'float64'>


In [ ]:
# Model Creation
seed = 42

tf.random.set_seed(seed)
input_layer = tfkl.Input(shape=input_shape, name='input_layer')

# Downsampling path
down_block_1 = unet_block(input_layer, 32, name='down_block1_')
d1 = tfkl.MaxPooling2D()(down_block_1)

down_block_2 = unet_block(d1, 64, name='down_block2_')
d2 = tfkl.MaxPooling2D()(down_block_2)

# Bottleneck
bottleneck = unet_block(d2, 128, name='bottleneck')

# Upsampling path
u1 = tfkl.UpSampling2D()(bottleneck)
u1 = tfkl.Concatenate()([u1, down_block_2])
u1 = unet_block(u1, 64, name='up_block1_')

u2 = tfkl.UpSampling2D()(u1)
u2 = tfkl.Concatenate()([u2, down_block_1])
u2 = unet_block(u2, 32, name='up_block2_')

# Output Layer
output_layer = tfkl.Conv2D(num_classes, kernel_size=1, padding='same', activation="softmax", name='output_layer')(u2)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer, name='UNet')

# Define the MeanIoU ignoring the background class
mean_iou = tfk.metrics.MeanIoU(num_classes=num_classes, ignore_class=0, sparse_y_pred=False)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=[mean_iou, "accuracy"])

model.summary()

Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 64, 128, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv1         │ (None, 64, 128, 32)    │            320 │ input_layer[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn1           │ (None, 64, 128, 32)    │            128 │ down_block1_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation1   │ (None, 64, 128, 32)    │              0 │ down_block1_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv2         │ (None, 64, 128, 32)    │          9,248 │ down_block1_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn2           │ (None, 64, 128, 32)    │            128 │ down_block1_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation2   │ (None, 64, 128, 32)    │              0 │ down_block1_bn2[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d             │ (None, 32, 64, 32)     │              0 │ down_block1_activatio… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv1         │ (None, 32, 64, 64)     │         18,496 │ max_pooling2d[0][0]    │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn1           │ (None, 32, 64, 64)     │            256 │ down_block2_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activation1   │ (None, 32, 64, 64)     │              0 │ down_block2_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv2         │ (None, 32, 64, 64)     │         36,928 │ down_block2_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn2           │ (None, 32, 64, 64)     │            256 │ down_block2_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activatio

 Total params: 473,669 (1.81 MB)

 Trainable params: 472,389 (1.80 MB)

 Non-trainable params: 1,280 (5.00 KB)

In [ ]:
# Setup callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=PATIENCE,
    restore_best_weights=True
)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
).history

Epoch 1/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 45s 541ms/step - accuracy: 0.3731 - loss: 1.4444 - mean_io_u: 0.1619 - val_accuracy: 0.1883 - val_loss: 2.5407 - val_mean_io_u: 0.0642
Epoch 2/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 109ms/step - accuracy: 0.5203 - loss: 1.1784 - mean_io_u: 0.2630 - val_accuracy: 0.1883 - val_loss: 3.7486 - val_mean_io_u: 0.0642
Epoch 3/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 121ms/step - accuracy: 0.5748 - loss: 1.0536 - mean_io_u: 0.3055 - val_accuracy: 0.1884 - val_loss: 3.4151 - val_mean_io_u: 0.0642
Epoch 4/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 110ms/step - accuracy: 0.5885 - loss: 1.0094 - mean_io_u: 0.3150 - val_accuracy: 0.1884 - val_loss: 3.2678 - val_mean_io_u: 0.0642
Epoch 5/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 116ms/step - accuracy: 0.6272 - loss: 0.9386 - mean_io_u: 0.3448 - val_accuracy: 0.1884 - val_loss: 3.0252 - val_mean_io_u: 0.0513
Epoch 6/1000
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - accuracy: 0.6227 - loss: 0.9515 - mean_io_u: 0.3401 - val_accuracy: 0.188

In [ ]:
timestep_str = datetime.now().strftime("%y%m%d_%H%M%S")
model_filename = f"model_{timestep_str}.keras"
model.save(model_filename)
del model

print(f"Model saved to {model_filename}")

Model saved to model_241130_150002.keras


# 03 - Addition of Basic Autoencoder whit Oversampled Dataset with Image Creation


## ⏳ Load the Data

In [ ]:
data = np.load("mars_for_students.npz")
train = np.load('oversampled_normalized_train_mars_for_students.npz')

X_train = train['images']
y_train = train['labels']

X_test = data["test_set"]

print(f"Training X shape: {X_train.shape}")
print(f"Training y shape: {y_train.shape}")
print(f"Test X shape: {X_test.shape}")

Training X shape: (3555, 64, 128, 1)
Training y shape: (3555, 64, 128, 1)
Test X shape: (10022, 64, 128)


## 🛠️ Train and Save the Model

In [ ]:
# Add color channel and rescale pixels between 0 and 1
X_test = X_test[..., np.newaxis] / 255.0

input_shape = X_train.shape[1:]
num_classes = len(np.unique(y_train))

print(f"Input shape: {input_shape}")
print(f"Number of classes: {num_classes}")

Input shape: (64, 128, 1)
Number of classes: 5


In [ ]:
# Set batch size for training
BATCH_SIZE = 64

# Set learning rate for the optimiser
LEARNING_RATE = 1e-3

# Set early stopping patience threshold
PATIENCE = 30

# Set maximum number of training epochs
EPOCHS = 1000

# Set data split size for training and validation
SPLITS_SIZE = 300

In [ ]:
train_img, val_img, train_lbl, val_lbl = train_test_split(
    X_train, y_train, test_size=300, random_state=42
)
print("Data splitted!")

print(f"\nNumber of images:")
print(f"Train: {len(train_img)}")
print(f"Validation: {len(val_img)}")
#print(f"Test: {len(test_img)}")

Data splitted!

Number of images:
Train: 3255
Validation: 300


### Train with Augmentation One (Only vertical and horizontal flips)

In [ ]:
# Create the datasets
print("Creating datasets...")
train_dataset = make_dataset_augOne(
    train_img, train_lbl,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    seed=42
)

val_dataset = make_dataset_augOne(
    val_img, val_lbl,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Datasets created!")

# Check the shape of the data
for images, labels in train_dataset.take(1):
    input_shape = images.shape[1:]
    print(f"\nInput shape: {input_shape}")
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Labels dtype:", labels.dtype)
    break

Creating datasets...
Datasets created!

Input shape: (64, 128, 1)
Images shape: (64, 64, 128, 1)
Labels shape: (64, 64, 128, 1)
Labels dtype: <dtype: 'int64'>


In [ ]:
# Model Creation
seed = 42

tf.random.set_seed(seed)
input_layer = tfkl.Input(shape=input_shape, name='input_layer')

# Downsampling path
down_block_1 = unet_block(input_layer, 32, name='down_block1_')
d1 = tfkl.MaxPooling2D()(down_block_1)

down_block_2 = unet_block(d1, 64, name='down_block2_')
d2 = tfkl.MaxPooling2D()(down_block_2)

# Bottleneck
bottleneck = unet_block(d2, 128, name='bottleneck')

# Upsampling path
u1 = tfkl.UpSampling2D()(bottleneck)
u1 = tfkl.Concatenate()([u1, down_block_2])
u1 = unet_block(u1, 64, name='up_block1_')

u2 = tfkl.UpSampling2D()(u1)
u2 = tfkl.Concatenate()([u2, down_block_1])
u2 = unet_block(u2, 32, name='up_block2_')

# Output Layer
output_layer = tfkl.Conv2D(num_classes, kernel_size=1, padding='same', activation="softmax", name='output_layer')(u2)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer, name='UNet')

# Define the MeanIoU ignoring the background class
mean_iou = tfk.metrics.MeanIoU(num_classes=num_classes, ignore_class=0, sparse_y_pred=False)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=[mean_iou, "accuracy"])

model.summary()

Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 64, 128, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv1         │ (None, 64, 128, 32)    │            320 │ input_layer[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn1           │ (None, 64, 128, 32)    │            128 │ down_block1_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation1   │ (None, 64, 128, 32)    │              0 │ down_block1_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv2         │ (None, 64, 128, 32)    │          9,248 │ down_block1_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn2           │ (None, 64, 128, 32)    │            128 │ down_block1_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation2   │ (None, 64, 128, 32)    │              0 │ down_block1_bn2[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_2           │ (None, 32, 64, 32)     │              0 │ down_block1_activatio… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv1         │ (None, 32, 64, 64)     │         18,496 │ max_pooling2d_2[0][0]  │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn1           │ (None, 32, 64, 64)     │            256 │ down_block2_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activation1   │ (None, 32, 64, 64)     │              0 │ down_block2_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv2         │ (None, 32, 64, 64)     │         36,928 │ down_block2_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn2           │ (None, 32, 64, 64)     │            256 │ down_block2_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activatio

 Total params: 473,669 (1.81 MB)

 Trainable params: 472,389 (1.80 MB)

 Non-trainable params: 1,280 (5.00 KB)

In [ ]:
# Setup callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=PATIENCE,
    restore_best_weights=True
)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
).history

Epoch 1/1000
51/51 ━━━━━━━━━━━━━━━━━━━━ 56s 525ms/step - accuracy: 0.3970 - loss: 1.4481 - mean_io_u_1: 0.1730 - val_accuracy: 0.2407 - val_loss: 2.0326 - val_mean_io_u_1: 0.0808
Epoch 2/1000
51/51 ━━━━━━━━━━━━━━━━━━━━ 39s 120ms/step - accuracy: 0.5450 - loss: 1.1300 - mean_io_u_1: 0.2686 - val_accuracy: 0.2407 - val_loss: 2.1855 - val_mean_io_u_1: 0.0646
Epoch 3/1000
51/51 ━━━━━━━━━━━━━━━━━━━━ 10s 120ms/step - accuracy: 0.5704 - loss: 1.0531 - mean_io_u_1: 0.2906 - val_accuracy: 0.2407 - val_loss: 2.2002 - val_mean_io_u_1: 0.0646
Epoch 4/1000
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 125ms/step - accuracy: 0.5876 - loss: 1.0165 - mean_io_u_1: 0.2984 - val_accuracy: 0.2407 - val_loss: 2.1986 - val_mean_io_u_1: 0.0646
Epoch 5/1000
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 122ms/step - accuracy: 0.6097 - loss: 0.9696 - mean_io_u_1: 0.3250 - val_accuracy: 0.2407 - val_loss: 2.6834 - val_mean_io_u_1: 0.0646
Epoch 6/1000
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 125ms/step - accuracy: 0.6131 - loss: 0.9588 - mean_io_u_1: 0.32

In [ ]:
timestep_str = datetime.now().strftime("%y%m%d_%H%M%S")
model_filename = f"model_{timestep_str}.keras"
model.save(model_filename)
del model

print(f"Model saved to {model_filename}")

Model saved to model_241202_095859.keras


# 03 - Addition of Basic Autoencoder by pure Reusage


## ⏳ Load the Data

In [ ]:
data = np.load("mars_for_students.npz")

training_set = data["training_set"]
X_train = training_set[:, 0]
y_train = training_set[:, 1]

X_test = data["test_set"]

print(f"Training X shape: {X_train.shape}")
print(f"Training y shape: {y_train.shape}")
print(f"Test X shape: {X_test.shape}")

Training X shape: (2615, 64, 128)
Training y shape: (2615, 64, 128)
Test X shape: (10022, 64, 128)


## 🛠️ Train and Save the Model

In [ ]:
# Add color channel and rescale pixels between 0 and 1
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

input_shape = X_train.shape[1:]
num_classes = len(np.unique(y_train))

print(f"Input shape: {input_shape}")
print(f"Number of classes: {num_classes}")

Input shape: (64, 128, 1)
Number of classes: 5


In [ ]:
y_train = y_train[..., np.newaxis]
y_train.shape

(2615, 64, 128, 1)

### Train with Augmentation One (Only vertical and horizontal flips)

In [ ]:
def make_dataset_augOne_balanced(X, y, batch_size=64, shuffle=False, augment=False, seed=None):
    """
    Create a memory-efficient TensorFlow dataset.
    """
    dataset_with4 = tf.data.Dataset.from_tensor_slices((X, y)).filter(lambda x, y: tf.math.reduce_any(tf.math.equal(y, 4)))
    dataset_with41 = dataset_with4
    dataset_with42 = dataset_with4
    dataset_with43 = dataset_with4
    dataset_with44 = dataset_with4
    dataset_with45 = dataset_with4
    dataset_with46 = dataset_with4
    dataset_with47 = dataset_with4
    dataset_with48 = dataset_with4
    dataset_with49 = dataset_with4
    dataset_with50 = dataset_with4
    dataset_with51 = dataset_with4
    dataset_with52 = dataset_with4
    dataset_with53 = dataset_with4
    dataset_with54 = dataset_with4
    dataset_with55 = dataset_with4
    dataset_with56 = dataset_with4
    dataset_with57 = dataset_with4
    dataset_with58 = dataset_with4
    dataset_with59 = dataset_with4
    dataset_with60 = dataset_with4
    dataset_with61 = dataset_with4
    dataset_with62 = dataset_with4
    dataset_with63 = dataset_with4
    dataset_with64 = dataset_with4
    dataset_rest = tf.data.Dataset.from_tensor_slices((X, y)).filter(lambda x, y: tf.math.reduce_any(tf.math.not_equal(y, 4)))
    dataset = tf.data.Dataset.sample_from_datasets(
        [dataset_with4, dataset_with41, dataset_with42, dataset_with43, dataset_with44,
         dataset_with45, dataset_with46, dataset_with47, dataset_with48, dataset_with49,
         dataset_with50, dataset_with51, dataset_with52, dataset_with53, dataset_with54,
         dataset_with55, dataset_with56, dataset_with57, dataset_with58, dataset_with59,
         dataset_with60, dataset_with61, dataset_with62, dataset_with63, dataset_with64,
         dataset_rest],
          weights=[0.016, 0.016, .016, .016, .016,
                    0.016, 0.016, .016, .016, .016,
                    0.016, 0.016, .016, .016, .016,
                    0.016, 0.016, .016, .016, .016,
                    0.016, 0.016, .016, .016, .016,
                    .5])

    if shuffle:
        dataset = dataset.shuffle(buffer_size=batch_size * 2, seed=seed)

    if augment:
        dataset = dataset.map(
            lambda x, y: random_horizontal_flip(x, y, seed=seed),
            num_parallel_calls=tf.data.AUTOTUNE
        )

        dataset = dataset.map(
            lambda x, y: random_vertical_flip(x, y, seed=seed),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # Batch the data
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
# Create the datasets
print("Creating datasets...")
train_dataset = make_dataset_augOne_balanced(
    train_img, train_lbl,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    seed=42
)

val_dataset = make_dataset_augOne_balanced(
    val_img, val_lbl,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Datasets created!")

# Check the shape of the data
for images, labels in train_dataset.take(1):
    input_shape = images.shape[1:]
    print(f"\nInput shape: {input_shape}")
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Labels dtype:", labels.dtype)
    break

Creating datasets...
Datasets created!

Input shape: (64, 128, 1)
Images shape: (64, 64, 128, 1)
Labels shape: (64, 64, 128, 1)
Labels dtype: <dtype: 'float64'>


In [ ]:
# Model Creation
seed = 42

tf.random.set_seed(seed)
input_layer = tfkl.Input(shape=input_shape, name='input_layer')

# Downsampling path
down_block_1 = unet_block(input_layer, 32, name='down_block1_')
d1 = tfkl.MaxPooling2D()(down_block_1)

down_block_2 = unet_block(d1, 64, name='down_block2_')
d2 = tfkl.MaxPooling2D()(down_block_2)

# Bottleneck
bottleneck = unet_block(d2, 128, name='bottleneck')

# Upsampling path
u1 = tfkl.UpSampling2D()(bottleneck)
u1 = tfkl.Concatenate()([u1, down_block_2])
u1 = unet_block(u1, 64, name='up_block1_')

u2 = tfkl.UpSampling2D()(u1)
u2 = tfkl.Concatenate()([u2, down_block_1])
u2 = unet_block(u2, 32, name='up_block2_')

# Output Layer
output_layer = tfkl.Conv2D(num_classes, kernel_size=1, padding='same', activation="softmax", name='output_layer')(u2)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer, name='UNet')

# Define the MeanIoU ignoring the background class
mean_iou = tfk.metrics.MeanIoU(num_classes=num_classes, ignore_class=0, sparse_y_pred=False)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=[mean_iou, "accuracy"])

model.summary()

Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 64, 128, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv1         │ (None, 64, 128, 32)    │            320 │ input_layer[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn1           │ (None, 64, 128, 32)    │            128 │ down_block1_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation1   │ (None, 64, 128, 32)    │              0 │ down_block1_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv2         │ (None, 64, 128, 32)    │          9,248 │ down_block1_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn2           │ (None, 64, 128, 32)    │            128 │ down_block1_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation2   │ (None, 64, 128, 32)    │              0 │ down_block1_bn2[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d             │ (None, 32, 64, 32)     │              0 │ down_block1_activatio… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv1         │ (None, 32, 64, 64)     │         18,496 │ max_pooling2d[0][0]    │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn1           │ (None, 32, 64, 64)     │            256 │ down_block2_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activation1   │ (None, 32, 64, 64)     │              0 │ down_block2_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv2         │ (None, 32, 64, 64)     │         36,928 │ down_block2_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn2           │ (None, 32, 64, 64)     │            256 │ down_block2_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activatio

 Total params: 473,669 (1.81 MB)

 Trainable params: 472,389 (1.80 MB)

 Non-trainable params: 1,280 (5.00 KB)

In [ ]:
# Setup callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=PATIENCE,
    restore_best_weights=True
)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
).history

Epoch 1/1000
     58/Unknown 51s 322ms/step - accuracy: 0.4172 - loss: 1.3682 - mean_io_u: 0.1755

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


58/58 ━━━━━━━━━━━━━━━━━━━━ 56s 418ms/step - accuracy: 0.4182 - loss: 1.3663 - mean_io_u: 0.1761 - val_accuracy: 0.1906 - val_loss: 2.3276 - val_mean_io_u: 0.0702
Epoch 2/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 45s 133ms/step - accuracy: 0.4933 - loss: 1.2102 - mean_io_u: 0.2183 - val_accuracy: 0.1906 - val_loss: 3.3200 - val_mean_io_u: 0.0702
Epoch 3/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 8s 129ms/step - accuracy: 0.5372 - loss: 1.0919 - mean_io_u: 0.2859 - val_accuracy: 0.1906 - val_loss: 5.1034 - val_mean_io_u: 0.0702
Epoch 4/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 131ms/step - accuracy: 0.5705 - loss: 1.0409 - mean_io_u: 0.3024 - val_accuracy: 0.1906 - val_loss: 4.2609 - val_mean_io_u: 0.0562
Epoch 5/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 8s 129ms/step - accuracy: 0.5816 - loss: 1.0015 - mean_io_u: 0.3198 - val_accuracy: 0.1906 - val_loss: 5.0059 - val_mean_io_u: 0.0562
Epoch 6/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 135ms/step - accuracy: 0.5938 - loss: 0.9936 - mean_io_u: 0.3447 - val_accuracy: 0.1915 - val_lo

In [ ]:
timestep_str = datetime.now().strftime("%y%m%d_%H%M%S")
model_filename = f"model_{timestep_str}.keras"
model.save(model_filename)
del model

print(f"Model saved to {model_filename}")

Model saved to model_241202_104013.keras


# 03 - Addition of Basic Autoencoder whith Oversampled Dataset by Image Generation and Image Reuse

## ⏳ Load the Data

In [ ]:
data = np.load("mars_for_students.npz")
train = np.load('oversampled_normalized_train_mars_for_students.npz')

X_train = train['images']
y_train = train['labels']

X_test = data["test_set"]

print(f"Training X shape: {X_train.shape}")
print(f"Training y shape: {y_train.shape}")
print(f"Test X shape: {X_test.shape}")

Training X shape: (3555, 64, 128, 1)
Training y shape: (3555, 64, 128, 1)
Test X shape: (10022, 64, 128)


## 🛠️ Train and Save the Model

In [ ]:
# Add color channel and rescale pixels between 0 and 1
X_test = X_test[..., np.newaxis] / 255.0

input_shape = X_train.shape[1:]
num_classes = len(np.unique(y_train))

print(f"Input shape: {input_shape}")
print(f"Number of classes: {num_classes}")

Input shape: (64, 128, 1)
Number of classes: 5


In [ ]:
# Set batch size for training
BATCH_SIZE = 64

# Set learning rate for the optimiser
LEARNING_RATE = 1e-3

# Set early stopping patience threshold
PATIENCE = 30

# Set maximum number of training epochs
EPOCHS = 1000

# Set data split size for training and validation
SPLITS_SIZE = 300

In [ ]:
train_img, val_img, train_lbl, val_lbl = train_test_split(
    X_train, y_train, test_size=300, random_state=42
)
print("Data splitted!")

print(f"\nNumber of images:")
print(f"Train: {len(train_img)}")
print(f"Validation: {len(val_img)}")

Data splitted!

Number of images:
Train: 3255
Validation: 300


### Train with Augmentation One (Only vertical and horizontal flips)

In [ ]:
def make_dataset_augOne_MixedApp(X, y, batch_size, shuffle=False, augment=False, seed=None):
    """
    Create a memory-efficient TensorFlow dataset.
    """
    # Create dataset from file paths
    dataset_with4 = tf.data.Dataset.from_tensor_slices((X, y)).filter(lambda x, y: tf.math.reduce_any(tf.math.equal(y, 4)))
    dataset_rest = tf.data.Dataset.from_tensor_slices((X, y)).filter(lambda x, y: tf.math.reduce_any(tf.math.not_equal(y, 4)))
    dataset = tf.data.Dataset.sample_from_datasets(
        [dataset_with4, dataset_rest],
        weights=[.35, .65])

    if shuffle:
        dataset = dataset.shuffle(buffer_size=batch_size * 2, seed=seed)

    if augment:
        dataset = dataset.map(
            lambda x, y: random_horizontal_flip(x, y, seed=seed),
            num_parallel_calls=tf.data.AUTOTUNE
        )

        dataset = dataset.map(
            lambda x, y: random_vertical_flip(x, y, seed=seed),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # Batch the data
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
# Create the datasets
print("Creating datasets...")
train_dataset = make_dataset_augOne_MixedApp(
    train_img, train_lbl,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    seed=42
)

val_dataset = make_dataset_augOne_MixedApp(
    val_img, val_lbl,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Datasets created!")

# Check the shape of the data
for images, labels in train_dataset.take(1):
    input_shape = images.shape[1:]
    print(f"\nInput shape: {input_shape}")
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Labels dtype:", labels.dtype)
    break

Creating datasets...
Datasets created!

Input shape: (64, 128, 1)
Images shape: (64, 64, 128, 1)
Labels shape: (64, 64, 128, 1)
Labels dtype: <dtype: 'int64'>


In [ ]:
seed = 42

tf.random.set_seed(seed)
input_layer = tfkl.Input(shape=input_shape, name='input_layer')

# Downsampling path
down_block_1 = unet_block(input_layer, 32, name='down_block1_')
d1 = tfkl.MaxPooling2D()(down_block_1)

down_block_2 = unet_block(d1, 64, name='down_block2_')
d2 = tfkl.MaxPooling2D()(down_block_2)

# Bottleneck
bottleneck = unet_block(d2, 128, name='bottleneck')

# Upsampling path
u1 = tfkl.UpSampling2D()(bottleneck)
u1 = tfkl.Concatenate()([u1, down_block_2])
u1 = unet_block(u1, 64, name='up_block1_')

u2 = tfkl.UpSampling2D()(u1)
u2 = tfkl.Concatenate()([u2, down_block_1])
u2 = unet_block(u2, 32, name='up_block2_')

# Output Layer
output_layer = tfkl.Conv2D(num_classes, kernel_size=1, padding='same', activation="softmax", name='output_layer')(u2)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer, name='UNet')

# Define the MeanIoU ignoring the background class
mean_iou = tfk.metrics.MeanIoU(num_classes=num_classes, ignore_class=0, sparse_y_pred=False)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=[mean_iou, "accuracy"])

model.summary()

Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 64, 128, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv1         │ (None, 64, 128, 32)    │            320 │ input_layer[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn1           │ (None, 64, 128, 32)    │            128 │ down_block1_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation1   │ (None, 64, 128, 32)    │              0 │ down_block1_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_conv2         │ (None, 64, 128, 32)    │          9,248 │ down_block1_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_bn2           │ (None, 64, 128, 32)    │            128 │ down_block1_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block1_activation2   │ (None, 64, 128, 32)    │              0 │ down_block1_bn2[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_2           │ (None, 32, 64, 32)     │              0 │ down_block1_activatio… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv1         │ (None, 32, 64, 64)     │         18,496 │ max_pooling2d_2[0][0]  │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn1           │ (None, 32, 64, 64)     │            256 │ down_block2_conv1[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activation1   │ (None, 32, 64, 64)     │              0 │ down_block2_bn1[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_conv2         │ (None, 32, 64, 64)     │         36,928 │ down_block2_activatio… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_bn2           │ (None, 32, 64, 64)     │            256 │ down_block2_conv2[0][… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ down_block2_activatio

 Total params: 473,669 (1.81 MB)

 Trainable params: 472,389 (1.80 MB)

 Non-trainable params: 1,280 (5.00 KB)

In [ ]:
# Setup callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=PATIENCE,
    restore_best_weights=True
)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
).history

Epoch 1/1000
     65/Unknown 33s 322ms/step - accuracy: 0.4163 - loss: 1.4065 - mean_io_u_1: 0.1784

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


65/65 ━━━━━━━━━━━━━━━━━━━━ 37s 375ms/step - accuracy: 0.4173 - loss: 1.4044 - mean_io_u_1: 0.1789 - val_accuracy: 0.2404 - val_loss: 2.3282 - val_mean_io_u_1: 0.0835
Epoch 2/1000
65/65 ━━━━━━━━━━━━━━━━━━━━ 19s 115ms/step - accuracy: 0.5409 - loss: 1.1271 - mean_io_u_1: 0.2489 - val_accuracy: 0.2404 - val_loss: 2.4339 - val_mean_io_u_1: 0.0835
Epoch 3/1000
65/65 ━━━━━━━━━━━━━━━━━━━━ 8s 118ms/step - accuracy: 0.5711 - loss: 1.0573 - mean_io_u_1: 0.2719 - val_accuracy: 0.2404 - val_loss: 2.9465 - val_mean_io_u_1: 0.0835
Epoch 4/1000
65/65 ━━━━━━━━━━━━━━━━━━━━ 10s 120ms/step - accuracy: 0.5824 - loss: 1.0314 - mean_io_u_1: 0.2800 - val_accuracy: 0.2404 - val_loss: 3.4528 - val_mean_io_u_1: 0.0668
Epoch 5/1000
65/65 ━━━━━━━━━━━━━━━━━━━━ 8s 118ms/step - accuracy: 0.5965 - loss: 1.0016 - mean_io_u_1: 0.3022 - val_accuracy: 0.2404 - val_loss: 3.1441 - val_mean_io_u_1: 0.0666
Epoch 6/1000
65/65 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - accuracy: 0.6183 - loss: 0.9696 - mean_io_u_1: 0.3175 - val_accu

In [ ]:
timestep_str = datetime.now().strftime("%y%m%d_%H%M%S")
model_filename = f"model_{timestep_str}.keras"
model.save(model_filename)
del model

print(f"Model saved to {model_filename}")

Model saved to model_241202_115214.keras


## 📊 Prepare Your Submission

In our Kaggle competition, submissions are made as `csv` files. To create a proper `csv` file, you need to flatten your predictions and include an `id` column as the first column of your dataframe. To maintain consistency between your results and our solution, please avoid shuffling the test set. The code below demonstrates how to prepare the `csv` file from your model predictions.




In [ ]:
# If model_filename is not defined, load the most recent model from Google Drive
if "model_filename" not in globals() or model_filename is None:
    files = [f for f in os.listdir('.') if os.path.isfile(f) and f.startswith('model_') and f.endswith('.keras')]
    files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    if files:
        model_filename = files[0]
    else:
        raise FileNotFoundError("No model files found in the current directory.")

In [ ]:
model = tfk.models.load_model(model_filename)
print(f"Model loaded from {model_filename}")

Model loaded from model_241202_115214.keras


In [ ]:
preds = model.predict(X_test)
preds = np.argmax(preds, axis=-1)
print(f"Predictions shape: {preds.shape}")

314/314 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step
Predictions shape: (10022, 64, 128)


In [ ]:
def y_to_df(y) -> pd.DataFrame:
    """Converts segmentation predictions into a DataFrame format for Kaggle."""
    n_samples = len(y)
    y_flat = y.reshape(n_samples, -1)
    df = pd.DataFrame(y_flat)
    df["id"] = np.arange(n_samples)
    cols = ["id"] + [col for col in df.columns if col != "id"]
    return df[cols]

In [ ]:
# Create and download the csv submission file
timestep_str = model_filename.replace("model_", "").replace(".keras", "")
submission_filename = f"submission_{timestep_str}.csv"
submission_df = y_to_df(preds)
submission_df.to_csv(submission_filename, index=False)

from google.colab import files
files.download(submission_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>